<a href="https://colab.research.google.com/github/Kinara2020/pyGAM/blob/main/CODE_COMPATIBILITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install pygam -q


In [7]:
import numpy as np
import scipy as sp
from scipy import stats, linalg
import pandas as pd
from pygam import LinearGAM

mcycle_url = "https://vincentarelbundock.github.io/Rdatasets/csv/MASS/mcycle.csv"
df = pd.read_csv(mcycle_url)
X = df[['times']].values
y = df['accel'].values

gam = LinearGAM().fit(X, y)

idxs = gam.terms.get_coef_indices(0)
cov = gam.statistics_["cov"][idxs][:, idxs]
coef = np.ravel(gam.coef_[idxs])
coef -= coef.mean()

inv_cov, rank = sp.linalg.pinv(cov, return_rank=True)
score = coef.T.dot(inv_cov).dot(coef)
score = score / rank
p_old = 1 - sp.stats.f.cdf(score, rank,
        gam.statistics_["n_samples"] - gam.statistics_["edof"])
print("OLD p-value:", p_old)

OLD p-value: 1.1102230246251565e-16


In [8]:
eigenvalues, eigenvectors = np.linalg.eigh(cov)
tol = 1e-12
threshold = eigenvalues[-1] * tol
mask = eigenvalues > threshold
rank_new = mask.sum()
eigenvalues_r = eigenvalues[mask]
eigenvectors_r = eigenvectors[:, mask]

score_new = np.sum((eigenvectors_r.T.dot(coef) ** 2) / eigenvalues_r)
score_new = score_new / rank_new
p_new = 1 - sp.stats.f.cdf(score_new, rank_new,
        gam.statistics_["n_samples"] - gam.statistics_["edof"])
print("NEW p-value:", p_new)

NEW p-value: 1.1102230246251565e-16


In [9]:
%load_ext rpy2.ipython

In [10]:
%%R
install.packages("mgcv", quiet=TRUE)
library(mgcv)
data(mcycle, package="MASS")
fit <- gam(accel ~ s(times), data=mcycle, method="REML")
summary(fit)


Family: gaussian 
Link function: identity 

Formula:
accel ~ s(times)

Parametric coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  -25.546      1.951  -13.09   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Approximate significance of smooth terms:
           edf Ref.df    F p-value    
s(times) 8.625  8.958 53.4  <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

R-sq.(adj) =  0.783   Deviance explained = 79.7%
-REML = 616.14  Scale est. = 506.35    n = 133


Loading required package: nlme
This is mgcv 1.9-4. For overview type '?mgcv'.


In [11]:
import numpy as np
from pygam import LinearGAM

# Weak signal dataset - where the bug actually shows
np.random.seed(42)
X_weak = np.linspace(0, 1, 50).reshape(-1, 1)
y_weak = np.random.normal(0, 1, 50)  # pure noise - true p should be ~0.5+

gam2 = LinearGAM().fit(X_weak, y_weak)

idxs2 = gam2.terms.get_coef_indices(0)
cov2 = gam2.statistics_["cov"][idxs2][:, idxs2]
coef2 = np.ravel(gam2.coef_[idxs2])
coef2 -= coef2.mean()

# OLD
inv_cov2, rank2 = sp.linalg.pinv(cov2, return_rank=True)
score2 = coef2.T.dot(inv_cov2).dot(coef2)
score2 = score2 / rank2
p_old2 = 1 - sp.stats.f.cdf(score2, rank2,
         gam2.statistics_["n_samples"] - gam2.statistics_["edof"])
print("OLD p-value (noise data):", p_old2)

# NEW
eigenvalues2, eigenvectors2 = np.linalg.eigh(cov2)
threshold2 = eigenvalues2[-1] * 1e-12
mask2 = eigenvalues2 > threshold2
rank_new2 = mask2.sum()
ev2 = eigenvalues2[mask2]
evec2 = eigenvectors2[:, mask2]
score_new2 = np.sum((evec2.T.dot(coef2) ** 2) / ev2)
score_new2 = score_new2 / rank_new2
p_new2 = 1 - sp.stats.f.cdf(score_new2, rank_new2,
         gam2.statistics_["n_samples"] - gam2.statistics_["edof"])
print("NEW p-value (noise data):", p_new2)
print("True expected p-value: ~0.5 (pure noise, no real effect)")

OLD p-value (noise data): 0.3352574371481294
NEW p-value (noise data): 0.3352574371481404
True expected p-value: ~0.5 (pure noise, no real effect)


In [12]:
# Force high smoothing - this creates the degenerate covariance that triggers the bug
from pygam import LinearGAM

gam3 = LinearGAM(lam=1e8).fit(X_weak, y_weak)  # extremely high smoothing

idxs3 = gam3.terms.get_coef_indices(0)
cov3 = gam3.statistics_["cov"][idxs3][:, idxs3]
coef3 = np.ravel(gam3.coef_[idxs3])
coef3 -= coef3.mean()

# OLD
inv_cov3, rank3 = sp.linalg.pinv(cov3, return_rank=True)
score3 = coef3.T.dot(inv_cov3).dot(coef3)
score3 = score3 / rank3
p_old3 = 1 - sp.stats.f.cdf(score3, rank3,
         gam3.statistics_["n_samples"] - gam3.statistics_["edof"])
print("OLD p-value (high smoothing):", p_old3)

# NEW
eigenvalues3, eigenvectors3 = np.linalg.eigh(cov3)
threshold3 = eigenvalues3[-1] * 1e-12
mask3 = eigenvalues3 > threshold3
rank_new3 = mask3.sum()
ev3 = eigenvalues3[mask3]
evec3 = eigenvectors3[:, mask3]
score_new3 = np.sum((evec3.T.dot(coef3) ** 2) / ev3)
score_new3 = score_new3 / rank_new3
p_new3 = 1 - sp.stats.f.cdf(score_new3, rank_new3,
         gam3.statistics_["n_samples"] - gam3.statistics_["edof"])
print("NEW p-value (high smoothing):", p_new3)
print("Rank kept (old vs new):", rank3, "vs", rank_new3)
print("Eigenvalue range:", eigenvalues3.min(), "to", eigenvalues3.max())

OLD p-value (high smoothing): 0.4135703940375579
NEW p-value (high smoothing): 0.35926153522068005
Rank kept (old vs new): 6 vs 3
Eigenvalue range: -4.0317735735953584e-17 to 0.45168540579595123
